### Inspired by Week 4, this notebook builds a coding tool that given a Python code, the user selects models, and languages to port to.

### The results are well formated, and the user has ability to add extra instructions. 

In [ ]:
import os
from openai import OpenAI
import gradio as gr
import re

Example Code

In [ ]:
python_code_example="""
import time

def func1(arr, temp, left, mid, right):
    i = left
    j = mid + 1
    k = left

    while i <= mid and j <= right:
        if arr[i] <= arr[j]:
            temp[k] = arr[i]
            i += 1
        else:
            temp[k] = arr[j]
            j += 1
        k += 1

    while i <= mid:
        temp[k] = arr[i]
        i += 1
        k += 1

    while j <= right:
        temp[k] = arr[j]
        j += 1
        k += 1

    for x in range(left, right + 1):
        arr[x] = temp[x]


def func2(arr, temp, left, right):
    if left >= right:
        return

    mid = (left + right) // 2

    func2(arr, temp, left, mid)
    func2(arr, temp, mid + 1, right)

    func1(arr, temp, left, mid, right)


def benchmark():
    start = time.time()

    n = 1_000_000
    arr = list(range(n, 0, -1))
    temp = [0] * n

    for _ in range(10):
        test = arr[:]
        func2(test, temp, 0, n - 1)

    elapsed = time.time() - start
    print(f"Time: {elapsed:.2f} seconds")


benchmark()
"""

In [ ]:
system_prompt = """
Your task is to convert Python code  from the user to its equivalent code in a given language.
Respond only with the code in the given language. Do not provide any explanation other than occasional comments.
The response needs to produce an identical output in the fastest possible time and memory. The code output should be well structured as a markdown code block.
"""


def user_prompt(language, python_code, instructions=None):
    extra = ""
    if instructions and instructions.strip():
        extra = f"\nUser instructions or context:\n{instructions.strip()}\n"
    return f"""
Convert the following Python code to {language} with the fastest possible implementation that produces identical output in the least time and memory.
{extra}
Python code to convert:

```python
{python_code}
```
"""

In [ ]:
ollama_url = "http://127.0.0.1:11434/v1"
ollama = OpenAI(api_key="ollama", base_url=ollama_url)
openrouter = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENAI_API_KEY"),
)

clients_model_mapping = {
    "gpt-5": openrouter,
    "anthropic/claude-sonnet-4.5": openrouter,
    "google/gemini-2.5-pro": openrouter,
    "qwen2.5-coder:1.5b": ollama,
    "deepseek-coder:1.3b": ollama,
}


def call_the_llm(model, prompt):
    client = clients_model_mapping[model]
    response = client.chat.completions.create(
        model=model,
        messages=prompt,
    )
    return response.choices[0].message.content

In [ ]:
# Rust not in gr.Code.languages; use markdown for display
LANG_TO_SYNTAX = {"C++": "cpp", "Javascript": "javascript", "Rust": "markdown"}


def extract_code_from_markdown(text):
    """Extract code from markdown fenced block if present."""
    if not text or not isinstance(text, str):
        return text or ""
    match = re.search(r"```(?:\w+)?\s*\n(.*?)```", text, re.DOTALL)
    return match.group(1).strip() if match else text.strip()


def convert_code(python_code, user_text, language, model):
    if not python_code:
        return ""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt(language, python_code, user_text)},
    ]
    try:
        response = call_the_llm(model, messages)
        return extract_code_from_markdown(response)
    except Exception as e:
        return f"Error: {e}"


with gr.Blocks(theme=gr.themes.Monochrome(), title="Port from Python to another language") as ui:
    with gr.Row(equal_height=False):
        with gr.Column(scale=6):
            user_text = gr.Textbox(
                label=" Additional Instructions or context (optional)",
                placeholder="e.g. Prefer readability, or add constraints...",
                value="",
                lines=2,
            )
            user_input = gr.Code(
                label="Python code",
                language="python",
                value="",
                lines=24,
            )

        with gr.Column(scale=2):
            language = gr.Dropdown(
                choices=["C++", "Javascript", "Rust"],
                value="C++",
                label="Language",
            )
            model = gr.Dropdown(
                choices=list(clients_model_mapping.keys()),
                value="gpt-5",
                label="Model",
            )
            convert_btn = gr.Button("Convert")
        with gr.Column(scale=6):
            generated = gr.Code(
                label="Generated",
                value="",
                language="cpp",
                lines=26
            )

    # When language changes, update the output code block label and syntax
    def update_output_ui(lang):
        syntax = LANG_TO_SYNTAX.get(lang, "cpp")
        return gr.update(label=f"{lang} generated", language=syntax)

    language.change(fn=update_output_ui, inputs=[language], outputs=[generated])

    # Convert button: run LLM and fill generated code
    convert_btn.click(
        fn=convert_code,
        inputs=[user_input, user_text, language, model],
        outputs=[generated],
    )

ui.launch()